In [1]:
import json

from datasets import load_dataset
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


## Dataset

In [2]:
repo_id = "Studeni/robot-instructions"
dataset = load_dataset(repo_id, split="test")

In [3]:
test_input = dataset[0]["input"]
test_output = dataset[0]["output"]
print(f"User input: {test_input}\nGround truth: {test_output}")

User input: Move TCP to (300, -100, 400) millimeters
Ground truth: [{"function": "move_tcp", "kwargs": {"x": 300.0, "y": -100.0, "z": 400.0}}]


## Prompt

In [4]:
robot_instruct_prompt = """
### Instruction:
Transform input into list of function calls for controlling industrial robots.

### Input:
{}

### Response:
{}
"""

## Model Parameters

In [5]:
lora_id = "Studeni/llama-3-8b-bnb-4bit-robot-instruct"
max_seq_length = 2048  # Choose any! Unsloth auto support RoPE Scaling internally!
dtype = (
    None  # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
)
load_in_4bit = True

## Run Model with Unsloth

Load the model and tokenizer

In [14]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=lora_id,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
FastLanguageModel.for_inference(model)

==((====))==  Unsloth: Fast Llama patching release 2024.6
   \\   /|    GPU: NVIDIA GeForce RTX 3090. Max memory: 23.562 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.3.0+cu121. CUDA = 8.6. CUDA Toolkit = 12.1.
\        /    Bfloat16 = TRUE. Xformers = 0.0.26.post1. FA = True.
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Unsloth 2024.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Tokenize input text

In [15]:
inputs = tokenizer(
    [
        robot_instruct_prompt.format(
            test_input,  # input
            "",  # output - leave this blank for generation!
        )
    ],
    return_tensors="pt",
).to("cuda")

Run generation

In [16]:
outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
text_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Extracting only the function call and convert it into json

In [17]:
function_call = text_output[0].split("### Response:")[-1].strip()
function_call = json.loads(function_call)

In [18]:
for f in function_call:
    print(f"Function to call: {f['function']}")
    print(f"Input paramaeters: {f['kwargs']}")

Function to call: move_tcp
Input paramaeters: {'x': 300.0, 'y': -100.0, 'z': 400.0}


## Run Model with Transformers and Peft

Load model and tokenizer

In [6]:
model = AutoPeftModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=lora_id,  # YOUR MODEL YOU USED FOR TRAINING
    load_in_4bit=load_in_4bit,
)
tokenizer = AutoTokenizer.from_pretrained(lora_id)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
`low_cpu_mem_usage` was None, now set to True since model is quantized.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Tokenize the input text

In [7]:
inputs = tokenizer(
    [
        robot_instruct_prompt.format(
            test_input,  # input
            "",  # output - leave this blank for generation!
        )
    ],
    return_tensors="pt",
).to("cuda")

Run generation

In [8]:
outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
text_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Extracting only the function call and convert it into json

In [9]:
function_call = text_output[0].split("### Response:")[-1].strip()
function_call = json.loads(function_call)

In [10]:
for f in function_call:
    print(f"Function to call: {f['function']}")
    print(f"Input paramaeters: {f['kwargs']}")

Function to call: move_tcp
Input paramaeters: {'x': 300.0, 'y': -100.0, 'z': 400.0}
